# 🚬 흡연 분류 AI 해커톤 - V4 (최종 최적화)

**V3 → V4 핵심 개선:**
- ⭐ K-Fold 교차 예측 (데이터 100% 활용)
- ⭐ Stacking 앙상블 (2단계 모델)
- ⭐ Feature Selection (상위 30개 피처)
- ⭐ 임계값 0.005 단위 초세밀 탐색
- ⭐ 3개 제출 파일 생성

**목표:** 0.76+ 달성

---

## 📌 STEP 1: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

## 📌 STEP 2: 데이터 로드

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"흡연자 비율: {train['label'].mean()*100:.2f}%")

# ID 처리
train_df = train.copy()
test_df = test.copy()

if 'ID' in test_df.columns:
    test_id = test_df['ID'].copy()
else:
    test_id = test_df['id'].copy() if 'id' in test_df.columns else None

train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

X = train_df.drop('label', axis=1)
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

feature_cols = X.columns.tolist()
print(f"원본 특성 수: {len(feature_cols)}")

## 📌 STEP 3: 피처 엔지니어링

In [ ]:
def create_features_v4(df):
    """V4 피처 엔지니어링"""
    df = df.copy()
    col_map = {c: c.lower() for c in df.columns}
    df_l = df.rename(columns=col_map)
    cols = df_l.columns.tolist()
    
    # 1. 콜레스테롤 관련
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_l['hdl'] / (df_l['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_l['ldl'] / (df_l['hdl'] + 1)
    if 'cholesterol' in cols and 'hdl' in cols:
        df['Atherogenic_idx'] = (df_l['cholesterol'] - df_l['hdl']) / (df_l['hdl'] + 1)
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df_l['triglyceride'] / (df_l['hdl'] + 1)
    
    # 2. 간 기능
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_l['gtp'])
        df['GTP_sq'] = df_l['gtp'] ** 2
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_l['ast'] / (df_l['alt'] + 1)
    
    # 3. 헤모글로빈
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_l['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_l['hemoglobin'])
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df_l['hemoglobin'] * df_l['gtp']
    
    # 4. 혈압
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_l['systolic'] - df_l['diastolic']
        df['MAP'] = df_l['diastolic'] + (df_l['systolic'] - df_l['diastolic']) / 3
    
    # 5. 체형
    if 'height' in cols and 'weight' in cols:
        df['BMI_calc'] = df_l['weight'] / ((df_l['height']/100) ** 2 + 0.01)
    
    # 6. 시력
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df_l[eye_cols].mean(axis=1)
    
    # 7. 나이 상호작용
    if 'age' in cols:
        age = df_l['age']
        df['Age_sq'] = age ** 2
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = age * df_l['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = age * df_l['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = age * df_l['triglyceride']
    
    # 8. 중성지방
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_l['triglyceride'])
    
    # 9. 혈당
    fbs_cols = [c for c in cols if 'blood' in c or 'fasting' in c]
    if len(fbs_cols) > 0:
        df['FBS_log'] = np.log1p(df_l[fbs_cols[0]])
    
    # 10. 건강 통계
    health_cols = [c for c in cols if c in ['systolic','diastolic','hemoglobin','triglyceride','cholesterol','hdl','gtp']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_l[health_cols].mean(axis=1)
        df['Health_std'] = df_l[health_cols].std(axis=1)
    
    df = df.fillna(0).replace([np.inf, -np.inf], 0)
    return df

X_fe = create_features_v4(X)
X_test_fe = create_features_v4(X_test)
print(f"피처 엔지니어링 후: {X_fe.shape[1]}개")

## 📌 STEP 4: Feature Selection (⭐ 상위 30개 선택)

In [ ]:
print("=" * 50)
print("🔍 Feature Selection: 중요 피처 선택")
print("=" * 50)

# 스케일링
scaler_temp = StandardScaler()
X_temp = scaler_temp.fit_transform(X_fe)

# LightGBM으로 피처 중요도 계산
lgb_temp = LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
lgb_temp.fit(X_temp, y)

# 피처 중요도
importance = pd.DataFrame({
    'feature': X_fe.columns,
    'importance': lgb_temp.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 20 중요 피처:")
print(importance.head(20).to_string(index=False))

# 상위 30개 피처 선택
TOP_N = 30
top_features = importance.head(TOP_N)['feature'].tolist()

X_selected = X_fe[top_features]
X_test_selected = X_test_fe[top_features]

print(f"\n✅ 선택된 피처 수: {len(top_features)}개")
print(f"선택된 피처: {top_features}")

In [ ]:
# 최종 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
X_test_scaled = scaler.transform(X_test_selected)

print(f"최종 Train: {X_scaled.shape}")
print(f"최종 Test: {X_test_scaled.shape}")

## 📌 STEP 5: K-Fold 교차 예측 (⭐ 데이터 100% 활용)

In [ ]:
print("=" * 50)
print("🎯 K-Fold 교차 예측 (데이터 100% 활용)")
print("=" * 50)

N_SPLITS = 5
SEEDS = [42, 123, 456, 789, 1004, 2024, 7777, 8888, 9999, 1234]

# OOF (Out-of-Fold) 예측 저장
oof_xgb = np.zeros(len(X_scaled))
oof_lgb = np.zeros(len(X_scaled))
oof_cat = np.zeros(len(X_scaled))
oof_rf = np.zeros(len(X_scaled))

# Test 예측 저장
test_xgb = np.zeros(len(X_test_scaled))
test_lgb = np.zeros(len(X_test_scaled))
test_cat = np.zeros(len(X_test_scaled))
test_rf = np.zeros(len(X_test_scaled))

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n--- Seed {seed} ({seed_idx+1}/{len(SEEDS)}) ---")
    
    kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled, y)):
        X_tr, X_va = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        # XGBoost
        xgb_m = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.02,
                              subsample=0.7, colsample_bytree=0.7,
                              random_state=seed, verbosity=0, use_label_encoder=False)
        xgb_m.fit(X_tr, y_tr)
        oof_xgb[val_idx] += xgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_xgb += xgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # LightGBM
        lgb_m = LGBMClassifier(n_estimators=500, max_depth=5, learning_rate=0.02,
                               subsample=0.7, colsample_bytree=0.7,
                               random_state=seed, verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        oof_lgb[val_idx] += lgb_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_lgb += lgb_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # CatBoost
        cat_m = CatBoostClassifier(n_estimators=500, max_depth=5, learning_rate=0.02,
                                   random_state=seed, verbose=0)
        cat_m.fit(X_tr, y_tr)
        oof_cat[val_idx] += cat_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_cat += cat_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # Random Forest
        rf_m = RandomForestClassifier(n_estimators=300, max_depth=15,
                                      random_state=seed, n_jobs=-1)
        rf_m.fit(X_tr, y_tr)
        oof_rf[val_idx] += rf_m.predict_proba(X_va)[:, 1] / len(SEEDS)
        test_rf += rf_m.predict_proba(X_test_scaled)[:, 1] / (N_SPLITS * len(SEEDS))
    
    # 중간 점수 출력
    temp_pred = (oof_xgb >= 0.5).astype(int)
    temp_pred[oof_xgb == 0] = 0  # 아직 예측 안 된 부분
    
print("\n✅ K-Fold 교차 예측 완료!")

In [ ]:
# 각 모델 OOF 성능 확인
print("\n📊 각 모델 OOF 성능:")
for name, oof in [('XGBoost', oof_xgb), ('LightGBM', oof_lgb), 
                   ('CatBoost', oof_cat), ('RF', oof_rf)]:
    pred = (oof >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    print(f"{name}: {acc:.5f}")

## 📌 STEP 6: Stacking 앙상블 (⭐ 2단계 모델)

In [ ]:
print("=" * 50)
print("🎯 Stacking 앙상블 (Meta 모델 학습)")
print("=" * 50)

# 1단계 모델 예측을 특성으로 결합
oof_stack = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_rf])
test_stack = np.column_stack([test_xgb, test_lgb, test_cat, test_rf])

print(f"Stacking 입력 shape: {oof_stack.shape}")

# 2단계: Meta 모델 (Logistic Regression)
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(oof_stack, y)

# Stacking 예측
oof_meta = meta_model.predict_proba(oof_stack)[:, 1]
test_meta = meta_model.predict_proba(test_stack)[:, 1]

print(f"\nMeta 모델 가중치: {meta_model.coef_[0]}")
print(f"(XGBoost, LightGBM, CatBoost, RF 순)")

In [ ]:
# 단순 평균 vs Stacking 비교
oof_simple = (oof_xgb + oof_lgb + oof_cat + oof_rf) / 4

print("\n📊 앙상블 방법 비교:")
for name, oof in [('단순 평균', oof_simple), ('Stacking', oof_meta)]:
    pred = (oof >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    print(f"{name}: Acc={acc:.5f}, F1={f1:.5f}")

## 📌 STEP 7: 최적 임계값 탐색 (⭐ 0.005 단위)

In [ ]:
print("=" * 50)
print("🔍 최적 임계값 탐색 (0.005 단위 초세밀)")
print("=" * 50)

# 단순 평균과 Stacking 둘 다 테스트
results = []

for thresh in np.arange(0.30, 0.60, 0.005):
    # 단순 평균
    pred_simple = (oof_simple >= thresh).astype(int)
    acc_simple = accuracy_score(y, pred_simple)
    
    # Stacking
    pred_stack = (oof_meta >= thresh).astype(int)
    acc_stack = accuracy_score(y, pred_stack)
    
    results.append({
        'threshold': thresh,
        'acc_simple': acc_simple,
        'acc_stack': acc_stack,
        'best': max(acc_simple, acc_stack),
        'method': '단순평균' if acc_simple > acc_stack else 'Stacking'
    })

results_df = pd.DataFrame(results)

# 상위 15개 출력
print("\n📊 Threshold별 성능 (상위 15개):")
print(results_df.nlargest(15, 'best')[['threshold', 'acc_simple', 'acc_stack', 'method']].to_string(index=False))

# 최적값 찾기
best_row = results_df.loc[results_df['best'].idxmax()]
best_threshold = best_row['threshold']
best_method = best_row['method']
best_acc = best_row['best']

print(f"\n🏆 최적 설정:")
print(f"   임계값: {best_threshold:.3f}")
print(f"   방법: {best_method}")
print(f"   Accuracy: {best_acc:.5f}")

In [ ]:
# 최적 방법 선택
if best_method == 'Stacking':
    final_oof = oof_meta
    final_test = test_meta
else:
    final_oof = oof_simple
    final_test = (test_xgb + test_lgb + test_cat + test_rf) / 4

print(f"\n선택된 방법: {best_method}")
print(f"Test 예측값 범위: {final_test.min():.4f} ~ {final_test.max():.4f}")

## 📌 STEP 8: 여러 임계값으로 3개 파일 생성 (⭐)

In [ ]:
print("=" * 50)
print("📝 3개 제출 파일 생성 (다른 임계값)")
print("=" * 50)

# 최적 임계값 근처 3개 선택
thresholds = [
    best_threshold - 0.02,  # 낮은 임계값 (흡연자 더 많이 예측)
    best_threshold,          # 최적 임계값
    best_threshold + 0.02   # 높은 임계값 (비흡연자 더 많이 예측)
]

file_paths = []

for i, thresh in enumerate(thresholds):
    # 예측
    pred = (final_test >= thresh).astype(int)
    
    # OOF 성능 확인
    oof_pred = (final_oof >= thresh).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    # 제출 파일 생성
    sub_df = submission.copy()
    sub_df['label'] = pred
    sub_df['label'] = sub_df['label'].astype(int)
    
    # 파일명
    thresh_str = f"{thresh:.3f}".replace('.', '')
    filename = f'submission_v4_t{thresh_str}.csv'
    filepath = result_path + filename
    
    # 저장
    sub_df.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    # 정보 출력
    n_smoking = (pred == 1).sum()
    smoking_ratio = n_smoking / len(pred) * 100
    
    marker = "⭐" if i == 1 else "  "
    print(f"\n{marker} 파일 {i+1}: {filename}")
    print(f"   임계값: {thresh:.3f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   흡연 예측: {n_smoking}명 ({smoking_ratio:.1f}%)")

print("\n✅ 3개 파일 생성 완료!")

In [ ]:
# 각 파일 미리보기
print("\n📋 제출 파일 미리보기 (최적 임계값):")
best_sub = pd.read_csv(file_paths[1])
display(best_sub.head(10))

print(f"\nlabel 타입: {best_sub['label'].dtype}")
print(f"label 값: {sorted(best_sub['label'].unique())}")

## 📌 STEP 9: 다운로드

In [ ]:
from google.colab import files

print("=" * 60)
print("📥 파일 다운로드")
print("=" * 60)

# 3개 파일 모두 다운로드
for filepath in file_paths:
    files.download(filepath)
    print(f"다운로드: {filepath.split('/')[-1]}")

print("\n" + "=" * 60)
print("🎉 V4 완료!")
print("=" * 60)

print(f"\n📊 최종 설정:")
print(f"   - 방법: {best_method}")
print(f"   - 최적 임계값: {best_threshold:.3f}")
print(f"   - OOF Accuracy: {best_acc:.5f}")
print(f"   - 사용 피처: {len(top_features)}개")
print(f"   - K-Fold: {N_SPLITS}")
print(f"   - Seeds: {len(SEEDS)}개")

print(f"\n📁 생성된 파일 (3개):")
for i, (filepath, thresh) in enumerate(zip(file_paths, thresholds)):
    marker = "👉" if i == 1 else "  "
    print(f"{marker} {filepath.split('/')[-1]} (threshold={thresh:.3f})")

print(f"\n💡 제출 전략:")
print(f"   1. 먼저 ⭐최적 파일 (t{thresholds[1]:.3f}) 제출")
print(f"   2. 점수 확인 후 필요시 다른 파일 제출")
print(f"\n🍀 행운을 빕니다!")